# Decay rate of the smallest eigenvalue of the normalized gcd kernel

Numerical investigation of the question posed in `notes/formalization-findings.md`, section 8.

**Question.** The kernel $K(m,n) = \gcd(m,n)/\sqrt{mn}$ is positive semidefinite — *proved* in
Lean as `normalizedGcdKernel_posSemidef` (`RiemannVenue/Kernels/GCD.lean`). Every finite section
$K_N = (K(m,n))_{1\le m,n\le N}$ therefore has $\lambda_{\min}(K_N) > 0$ (the matrix is in fact
positive definite: its Smith determinant is $\prod_{n\le N}\varphi(n)/n \ne 0$). But
$\lambda_{\min}(K_N)$ decays as $N$ grows. **At what rate, and why?**

Everything below is floating-point evidence, not proof. Companion note with fits, literature and a
conjecture candidate: `notes/lambda-min-rate.md`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt


def gcd_kernel(n: int) -> np.ndarray:
    """K_N with entries gcd(m, k) / sqrt(m k), 1 <= m, k <= n."""
    i = np.arange(1, n + 1)
    g = np.gcd.outer(i, i).astype(float)
    return g / np.sqrt(np.outer(i, i))


def phi_sieve(n: int) -> np.ndarray:
    """Euler phi(k) for 0 <= k <= n (phi[0] unused)."""
    phi = np.arange(n + 1)
    for p in range(2, n + 1):
        if phi[p] == p:  # p prime
            phi[p::p] -= phi[p::p] // p
    return phi


def omega_sieve(n: int) -> np.ndarray:
    """Omega(k) = number of prime factors with multiplicity, 0 <= k <= n."""
    om = np.zeros(n + 1, dtype=np.int64)
    for p in range(2, n + 1):
        if om[p] == 0:  # p prime
            pk = p
            while pk <= n:
                om[pk::pk] += 1
                pk *= p
    return om


def squarefree_sieve(n: int) -> np.ndarray:
    sf = np.ones(n + 1, dtype=bool)
    p = 2
    while p * p <= n:
        if all(p % q for q in range(2, int(p ** 0.5) + 1)):
            sf[p * p::p * p] = False
        p += 1
    return sf

## 1. Eigenvalue scan over a log-spaced grid

Grid: $N = 25 \cdot 2^{k/2}$, $k = 0,\dots,18$ (ratio $\sqrt2$, up to $N = 12800$). For each $N$
we record $\lambda_{\min}$, $\lambda_{\max}$, and the counting of eigenvalues below fixed
thresholds (to see whether small eigenvalues are an edge effect or a fixed fraction of the
spectrum). Full `float64` `eigvalsh`; the $N=1600$ spectrum is kept for the determinant
cross-check below.

In [2]:
Ns = [int(round(x)) for x in np.geomspace(25, 12800, 19)]
scan = {}
spec_1600 = None
for N in Ns:
    vals = np.linalg.eigvalsh(gcd_kernel(N))
    assert vals[0] > 0, f"lambda_min not positive at N={N}"
    scan[N] = dict(lmin=vals[0], lmax=vals[-1],
                   c01=int(np.sum(vals < 0.1)), c02=int(np.sum(vals < 0.2)),
                   c05=int(np.sum(vals < 0.5)))
    if N == 1600:
        spec_1600 = vals.copy()

print(f"{'N':>6} {'lmin':>10} {'lmax':>9} {'#<0.1':>6} {'#<0.2':>6} {'#<0.5':>6}"
      f" {'frac<0.1':>9} {'frac<0.5':>9}")
for N in Ns:
    s = scan[N]
    print(f"{N:>6} {s['lmin']:>10.6f} {s['lmax']:>9.3f} {s['c01']:>6} {s['c02']:>6}"
          f" {s['c05']:>6} {s['c01']/N:>9.4f} {s['c05']/N:>9.4f}")

     N       lmin      lmax  #<0.1  #<0.2  #<0.5  frac<0.1  frac<0.5
    25   0.069794     6.395      1      4     11    0.0400    0.4400
    35   0.055263     7.273      2      6     16    0.0571    0.4571
    50   0.048642     8.469      4      8     22    0.0800    0.4400
    71   0.041001     9.588      5     12     31    0.0704    0.4366
   100   0.036355    10.936      6     17     44    0.0600    0.4400
   141   0.032399    12.292      8     25     63    0.0567    0.4468
   200   0.029292    13.840     13     34     89    0.0650    0.4450
   283   0.025749    15.436     18     48    127    0.0636    0.4488
   400   0.023250    17.215     24     66    179    0.0600    0.4475
   566   0.020703    19.066     37     92    254    0.0654    0.4488
   800   0.018566    21.105     50    131    360    0.0625    0.4500
  1131   0.016890    23.244     70    186    508    0.0619    0.4492
  1600   0.015287    25.575    102    265    717    0.0638    0.4481
  2263   0.014061    28.019    145

The fractions of eigenvalues below fixed thresholds converge (e.g. $\#\{\lambda < 0.1\}/N
\to \approx 0.064$, $\#\{\lambda < 0.5\}/N \to \approx 0.45$): the *bulk* spectral
distribution stabilizes, while the extreme edge $\lambda_{\min}$ keeps creeping toward $0$. The
decay of $\lambda_{\min}$ is an edge phenomenon of a converging spectral profile, not a growing
cluster at $0$.

## 2. Competing decay laws

Candidates for $\lambda_{\min}(N)$: (a) $c\,N^{-c_2}$; (b) $c\,(\log N)^{-c_2}$;
(c) $c_1/\log N$; (d) $c\,e^{-c_2\sqrt{\log N}}$. Each is linear in $\log \lambda_{\min}$
against a different regressor. Fits are over the larger-$N$ half of the grid ($N \ge 566$) to
reduce small-$N$ transient bias; full-range fits shown for contrast.

In [3]:
Narr = np.array(Ns, float)
lam = np.array([scan[N]['lmin'] for N in Ns])
lmax = np.array([scan[N]['lmax'] for N in Ns])
L = np.log(Narr)
Y = np.log(lam)


def linfit(X, Y):
    A = np.vstack([X, np.ones_like(X)]).T
    coef, *_ = np.linalg.lstsq(A, Y, rcond=None)
    resid = Y - A @ coef
    rms = float(np.sqrt(np.mean(resid ** 2)))
    n = len(X)
    se = rms / np.sqrt(np.sum((X - X.mean()) ** 2)) * np.sqrt(n / (n - 2))
    return coef[0], se, coef[1], rms


half = Narr >= 566
for name, X in [("(a) power        N^-c      ", L),
                ("(b) log-power    (log N)^-c", np.log(L)),
                ("(d) sqrt-exp  exp(-c sqrt(log N))", np.sqrt(L))]:
    s_h, se_h, b_h, r_h = linfit(X[half], Y[half])
    s_f, se_f, b_f, r_f = linfit(X, Y)
    print(f"{name}: top half c = {-s_h:.4f} +/- {se_h:.4f}  rms = {r_h:.5f}"
          f"   | full range c = {-s_f:.4f} +/- {se_f:.4f}  rms = {r_f:.5f}")

print("\ndiagnostic sequences (constant iff the law holds):")
print("lam * log N        :", np.round(lam * L, 4), " -> law (c) fails (not constant)")
print("lam * (log N)^2    :", np.round(lam * L ** 2, 4))
print("lam * e^{1.47 sqrt}:", np.round(lam * np.exp(1.472 * np.sqrt(L)), 4))
print("lam_min * lam_max  :", np.round(lam * lmax, 4))
lmax_c, lmax_se, lmax_b, _ = linfit(np.log(L[half]), np.log(lmax[half]))
print(f"\nlam_max ~ (log N)^c: c = {-lmax_c:.3f} +/- {lmax_se:.3f},"
      f" prefactor ~ {np.exp(lmax_b):.3f}")

(a) power        N^-c      : top half c = 0.2628 +/- 0.0034  rms = 0.00946   | full range c = 0.3130 +/- 0.0080  rms = 0.06233
(b) log-power    (log N)^-c: top half c = 2.0532 +/- 0.0182  rms = 0.00657   | full range c = 1.8581 +/- 0.0224  rms = 0.02963
(d) sqrt-exp  exp(-c sqrt(log N)): top half c = 1.4720 +/- 0.0071  rms = 0.00355   | full range c = 1.5452 +/- 0.0152  rms = 0.02415

diagnostic sequences (constant iff the law holds):
lam * log N        : [0.2247 0.1965 0.1903 0.1748 0.1674 0.1603 0.1552 0.1454 0.1393 0.1312
 0.1241 0.1187 0.1128 0.1086 0.1028 0.0987 0.0939 0.0896 0.0859]  -> law (c) fails (not constant)
lam * (log N)^2    : [0.7232 0.6985 0.7444 0.745  0.771  0.7935 0.8223 0.8207 0.8346 0.8318
 0.8296 0.8349 0.8321 0.839  0.8295 0.8311 0.8233 0.8162 0.8128]
lam * e^{1.47 sqrt}: [0.979  0.8868 0.8942 0.8564 0.8559 0.8564 0.8675 0.8506 0.8535 0.8424
 0.8347 0.837  0.8332 0.8409 0.834  0.8395 0.8369 0.8361 0.8403]
lam_min * lam_max  : [0.4463 0.4019 0.4119 0.3931 0.3976 

**Reading.** Law (c) $c_1/\log N$ fails outright ($\lambda \log N$ drops steadily). The pure
power (a) has visibly drifting local exponent (from $\approx -0.35$ to $\approx -0.23$) and the
worst residuals — the decay is slower than any fixed power of $N$. The two survivors are
logarithmic-type: (b) $(\log N)^{-c}$ with $c \approx 2.05$, and (d) $e^{-c\sqrt{\log N}}$
with $c \approx 1.47$; over $566 \le N \le 12800$ law (d) has the smallest residuals, while
$\lambda_{\min} \cdot (\log N)^2$ is constant to within $\pm 3\%$ over the whole range and
peaks near $N \approx 2300$. The window is too short to separate $(\log N)^{-2}$ with
lower-order corrections from $e^{-c\sqrt{\log N}}$. Meanwhile
$\lambda_{\max} \approx 0.47 (\log N)^2$ (exponent $2.03 \pm 0.03$) and the product
$\lambda_{\min}\lambda_{\max} \approx 0.390$ is constant to $\pm 1\%$ for $N \ge 283$ — the
two spectral edges appear to move reciprocally.

## 3. Determinant identity (Smith) and the geometric mean

Smith's determinant gives $\det K_N = \prod_{n \le N} \varphi(n)/n$ exactly, so
$\log\det K_N = \sum_{n\le N}\log(\varphi(n)/n)$ and the mean of $\log(\varphi(n)/n)$ tends
to $\sum_p \log(1-1/p)/p$. The geometric-mean eigenvalue therefore tends to a *positive*
constant: the spectrum does not collapse; only the edge does.

In [4]:
M = 10 ** 6
phiM = phi_sieve(M)
# log(phi(n)/n) = sum_{p|n} log(1 - 1/p); sieve directly over primes
isp = np.ones(M + 1, dtype=bool); isp[:2] = False
for q in range(2, int(M ** 0.5) + 1):
    if isp[q]:
        isp[q * q::q] = False
primes = np.flatnonzero(isp)
lg = np.zeros(M + 1)
for p in primes:
    lg[p::p] += np.log1p(-1.0 / p)
logdet = np.cumsum(lg[1:])

for N in (50, 200, 1600, 10 ** 4, 10 ** 5, 10 ** 6):
    print(f"N = {N:>8}: log det K_N = {logdet[N - 1]:>14.4f}   mean = {logdet[N - 1] / N:.6f}")

c_mean = float(np.sum(np.log1p(-1.0 / primes) / primes))
print(f"\nsum_p log(1-1/p)/p over p < 1e6 = {c_mean:.6f} (tail O(1e-7))")
print(f"geometric-mean eigenvalue -> exp = {np.exp(c_mean):.6f}")

# cross-check against the computed spectrum at N = 1600
print(f"\nN=1600 cross-check: sum log lambda_i = {np.sum(np.log(spec_1600)):.6f}"
      f"  vs  sieve = {logdet[1599]:.6f}")

N =       50: log det K_N =       -28.2313   mean = -0.564626
N =      200: log det K_N =      -115.1038   mean = -0.575519
N =     1600: log det K_N =      -927.3248   mean = -0.579578
N =    10000: log det K_N =     -5799.6423   mean = -0.579964
N =   100000: log det K_N =    -58004.7171   mean = -0.580047
N =  1000000: log det K_N =   -580057.5185   mean = -0.580058

sum_p log(1-1/p)/p over p < 1e6 = -0.580058 (tail O(1e-7))
geometric-mean eigenvalue -> exp = 0.559866

N=1600 cross-check: sum log lambda_i = -927.324755  vs  sieve = -927.324755


## 4. Structure: Gram factorization, inverse matrix, sieve bound

The Lean-proved factorization $K_N = D B \,\mathrm{diag}(\varphi)\, B^T D$ ($D =
\mathrm{diag}(1/\sqrt n)$, $B$ the divisibility 0/1 matrix) gives $K_N = C_N C_N^T$ with $C_N =
D B \,\mathrm{diag}(\varphi)^{1/2}$, so $\lambda_{\min}(K_N) = \sigma_{\min}(C_N)^2$: the
question is about the smallest singular value of the (lower-triangular) divisibility structure.

Inverting instead: $K_N^{-1} = D^{-1} M^T \mathrm{diag}(1/\varphi) M D^{-1}$ with $M = B^{-1}$
the Möbius matrix, and $1/\lambda_{\min}(K_N) = \lambda_{\max}(K_N^{-1})$. The $(1,1)$ entry is
a classical Selberg-sieve sum: $(K_N^{-1})_{11} = \sum_{k \le N} \mu^2(k)/\varphi(k) = \log N
+ A + o(1)$. Pinning $x_1 = 1$ gives the *rigorous* upper bound $\lambda_{\min}(K_N) \le
1/\sum_{k\le N}\mu^2(k)/\varphi(k) \approx 1/(\log N + 1.33)$; the observed
$\lambda_{\min}$ sits a further $\log$-factor below this one-coordinate sieve bound.

In [5]:
# Gram factorization check at N = 500
N = 500
i = np.arange(1, N + 1)
phiN = phi_sieve(N)[1:].astype(float)
B = (np.mod.outer(i, i) == 0).astype(float)          # B[m,d] = 1 iff d | m
C = (B / np.sqrt(i)[:, None]) * np.sqrt(phiN)[None, :]
K = gcd_kernel(N)
print("||K - C C^T||_max      =", np.abs(K - C @ C.T).max())
sv = np.linalg.svd(C, compute_uv=False)
print(f"sigma_min(C)^2         = {sv[-1] ** 2:.12g}")
print(f"lambda_min(K)          = {np.linalg.eigvalsh(K)[0]:.12g}")

# inverse-matrix identities at N = 1000
N = 1000
Kinv = np.linalg.inv(gcd_kernel(N))
sf = squarefree_sieve(N)[1:]
musq_phi = np.where(sf, 1.0 / phi_sieve(N)[1:], 0.0)
print(f"\nN=1000: (K^-1)_11 = {Kinv[0, 0]:.6f}   sum mu^2/phi = {musq_phi.sum():.6f}")
print(f"        max diag of K^-1 at n = {1 + int(np.argmax(np.diag(Kinv)))}")
print(f"        lambda_max(K^-1) = {np.linalg.eigvalsh(Kinv)[-1]:.6f}"
      f"   1/lambda_min(K) = {1 / np.linalg.eigvalsh(gcd_kernel(N))[0]:.6f}")

# the sieve constant A: sum_{k<=x} mu^2/phi - log x
sfM = squarefree_sieve(M)[1:]
cums = np.cumsum(np.where(sfM, 1.0 / phiM[1:], 0.0))
for x in (10 ** 3, 10 ** 4, 10 ** 5, 10 ** 6):
    print(f"x = {x:>8}: sum mu^2/phi - log x = {cums[x - 1] - np.log(x):.6f}")

print("\nsieve upper bound vs truth:")
for N in (100, 800, 6400):
    ub = 1.0 / cums[N - 1]
    print(f"N = {N:>5}: 1/(sum mu^2/phi) = {ub:.5f}   lambda_min = {scan[N]['lmin']:.5f}"
          f"   ratio = {ub / scan[N]['lmin']:.3f}")

||K - C C^T||_max      = 4.440892098500626e-16
sigma_min(C)^2         = 0.0213863231721
lambda_min(K)          = 0.0213863231721

N=1000: (K^-1)_11 = 8.240046   sum mu^2/phi = 8.240046
        max diag of K^-1 at n = 6
        lambda_max(K^-1) = 57.274735   1/lambda_min(K) = 57.274735
x =     1000: sum mu^2/phi - log x = 1.332290
x =    10000: sum mu^2/phi - log x = 1.333002
x =   100000: sum mu^2/phi - log x = 1.332588
x =  1000000: sum mu^2/phi - log x = 1.332574

sieve upper bound vs truth:
N =   100: 1/(sum mu^2/phi) = 0.16919   lambda_min = 0.03636   ratio = 4.654
N =   800: 1/(sum mu^2/phi) = 0.12468   lambda_min = 0.01857   ratio = 6.716
N =  6400: 1/(sum mu^2/phi) = 0.09905   lambda_min = 0.01072   ratio = 9.241


## 5. Where does the minimizing eigenvector live?

Two scenarios for the decay: an *edge effect of truncation* (mass on integers near $N$, e.g. large
primes, which see only a sliver of their divisor structure) or *intrinsic spectral accumulation*
(mass on structured integers deep inside the interval). We compute the eigenvector at
$N = 2000$.

In [6]:
N = 2000
vals2000, vecs = np.linalg.eigh(gcd_kernel(N))
v = vecs[:, 0]
v = v * np.sign(v[0])          # normalize overall sign so v_1 > 0
w = v ** 2
om = omega_sieve(N)[1:]
liou = np.where(om % 2 == 0, 1, -1)

print(f"lambda_min(2000) = {vals2000[0]:.6f}; next: {np.round(vals2000[1:5], 5)}")
top = np.argsort(-w)[:20]
print("\ntop-20 components:  n    v_n      |v_n|^2   Liouville(n)  sign match")
for j in top:
    n = j + 1
    match = "yes" if np.sign(v[j]) == liou[j] else "NO"
    print(f"{n:>20} {v[j]:>+9.4f} {w[j]:>9.5f} {liou[j]:>8}      {match}")

n_ax = np.arange(1, N + 1)
print()
print(f"mass on n <= 100      : {w[:100].sum():.4f}")
print(f"mass on n <= N/2      : {w[:N // 2].sum():.4f}")
print(f"mass on n >  N/2      : {w[N // 2:].sum():.6f}")
print(f"mass on primes > N/2  : {w[(om == 1) & (n_ax > N // 2)].sum():.2e}")
agree = w[np.sign(v) == liou].sum()
print(f"mass-weighted agreement of sign(v_n) with Liouville lambda(n): {agree:.6f}")

lambda_min(2000) = 0.014471; next: [0.0211  0.02362 0.02549 0.02669]

top-20 components:  n    v_n      |v_n|^2   Liouville(n)  sign match
                   6   +0.3027   0.09160        1      yes
                  30   -0.2589   0.06702       -1      yes
                   2   -0.2266   0.05136       -1      yes
                  12   -0.2173   0.04722       -1      yes
                  10   +0.2005   0.04020        1      yes
                  42   -0.1854   0.03436       -1      yes
                  60   +0.1785   0.03186        1      yes
                   3   -0.1770   0.03131       -1      yes
                   4   +0.1673   0.02799        1      yes
                  18   -0.1579   0.02494       -1      yes
                  15   +0.1553   0.02412        1      yes
                  14   +0.1458   0.02127        1      yes
                  20   -0.1432   0.02051       -1      yes
                   1   +0.1307   0.01708        1      yes
                 210   +0.1305   0.

The minimizer concentrates on *small, highly divisible (smooth) integers* — $n = 6, 30, 2, 12,
10, 42, 60, \dots$ — with $\approx 78\%$ of its mass on $n \le 100$ and $\approx 1\%$ on the
entire upper half of the interval; the mass on large primes is $\sim 10^{-7}$. And its signs
follow the Liouville function $\lambda(n) = (-1)^{\Omega(n)}$ essentially exactly (mass-weighted
agreement $> 0.999$). The decay of $\lambda_{\min}$ is **intrinsic spectral accumulation driven
by multiplicative structure at the small-$n$ end**, not a truncation edge effect: the minimizer is
a Liouville-signed, smooth-number-supported vector — the finite-rank shadow of a $1/\zeta$-type
mollifier.

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))

# (i) lambda_min vs N with fitted laws
ax = axes[0]
ax.loglog(Narr, lam, "o", color="k", ms=4, label=r"$\lambda_{\min}(K_N)$")
Ngrid = np.geomspace(25, 12800, 200)
Lg = np.log(Ngrid)
sb, _, bb, _ = linfit(np.log(L[half]), Y[half])
sd, _, bd, _ = linfit(np.sqrt(L[half]), Y[half])
sa, _, ba, _ = linfit(L[half], Y[half])
ax.loglog(Ngrid, np.exp(bb) * Lg ** sb, "-", lw=1.2,
          label=rf"$(\log N)^{{{sb:.2f}}}$ fit")
ax.loglog(Ngrid, np.exp(bd) * np.exp(sd * np.sqrt(Lg)), "--", lw=1.2,
          label=rf"$e^{{{sd:.2f}\sqrt{{\log N}}}}$ fit")
ax.loglog(Ngrid, np.exp(ba) * Ngrid ** sa, ":", lw=1.2, color="gray",
          label=rf"$N^{{{sa:.2f}}}$ fit (rejected)")
ax.set_xlabel("N"); ax.set_ylabel(r"$\lambda_{\min}$")
ax.set_title("Smallest eigenvalue: log-type decay")
ax.legend(fontsize=8)

# (ii) diagnostics
ax = axes[1]
ax.semilogx(Narr, lam * L ** 2, "o-", ms=4, label=r"$\lambda_{\min}\cdot(\log N)^2$")
ax.semilogx(Narr, lam * np.exp(1.472 * np.sqrt(L)), "s-", ms=4,
            label=r"$\lambda_{\min}\cdot e^{1.472\sqrt{\log N}}$")
ax.semilogx(Narr, lam * lmax, "^-", ms=4,
            label=r"$\lambda_{\min}\cdot\lambda_{\max}$")
ax.axhline(np.pi ** 2 / 12, color="gray", lw=0.8, ls=":")
ax.text(30, np.pi ** 2 / 12 + 0.01, r"$\pi^2/12$", color="gray", fontsize=8)
ax.set_xlabel("N"); ax.set_ylim(0, 1.05)
ax.set_title("Compensated sequences (constant = law holds)")
ax.legend(fontsize=8, loc="center right")

# (iii) minimizer mass profile at N = 2000
ax = axes[2]
n_ax = np.arange(1, 2001)
pos = v > 0
ax.scatter(n_ax[pos], w[pos], s=8, color="tab:blue", label=r"$v_n > 0$")
ax.scatter(n_ax[~pos], w[~pos], s=8, color="tab:red", label=r"$v_n < 0$")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_ylim(1e-12, 0.3)
for j in np.argsort(-w)[:6]:
    ax.annotate(str(j + 1), (j + 1, w[j]), textcoords="offset points",
                xytext=(2, 3), fontsize=8)
ax.set_xlabel("n"); ax.set_ylabel(r"$|v_n|^2$")
ax.set_title("Minimizing eigenvector, N = 2000\n(mass on small smooth n, Liouville signs)")
ax.legend(fontsize=8, loc="lower left")

fig.tight_layout()
fig.savefig("../figures/lambda-min-rate.png", dpi=150)
plt.show()

<Figure size 1500x460 with 3 Axes>

**Interpretation.** Numerically (evidence, not proof): $\lambda_{\min}(K_N)$ decays like a
power of $\log N$ — compatible with $c/(\log N)^2$, $c \approx 0.82$–$0.84$ over the computed
window (curiously close to $\pi^2/12 = 0.8225$, unverified numerology), with the alternative
$e^{-c\sqrt{\log N}}$, $c \approx 1.47$, fitting the top of the range slightly better; the data
cannot separate the two. Pure power laws in $N$ and $c_1/\log N$ are rejected. The mechanism is
intrinsic: the minimizer is a Liouville-signed vector supported on smooth numbers (a mollifier
for $1/\zeta$), the one-coordinate sieve pin gives the exact bound
$\lambda_{\min} \le 1/(\log N + A)$, and spreading over smooth $n$ buys roughly one more
$\log$ factor. Meanwhile $\det K_N = \prod\varphi(n)/n$ forces the geometric-mean eigenvalue to
the constant $e^{-0.580} \approx 0.56$: the bulk stays put while the two edges move reciprocally
($\lambda_{\min}\lambda_{\max} \approx 0.39$). See `notes/lambda-min-rate.md` for the
conjecture candidate and literature placement.